# VisionDrive: train bottle, cup, and box detection

An end-to-end notebook for the project's **ESPDet-Pico** detector and three-region instruction visualization.

**Exactly three classes:** `0 = bottle`, `1 = cup`, `2 = box`. This historical checkpoint excludes other categories. The current default viewer uses a broader laptop model; run --detector esp to compare this checkpoint. The controller consumes detections; it does not learn motor commands.

The notebook reuses the same preparation, training, and evaluation scripts as the desktop program. The published checkpoint/report can be inspected without repeating the training run. Enable the switches below to reproduce training.

This is a small educational prototype. Held-out public-image results do not establish performance on your camera, a mini-car course, or ESP32 hardware.


## 1. Select a Python kernel and locate the repository

Open this notebook from a clone of **Vision-Drive**. Select a kernel with the model dependencies in `requirements-model.txt` installed. The development run used Windows, Python 3.13.5, PyTorch 2.12.0 CPU, torchvision 0.27.0, and Ultralytics 8.3.112.

On the development computer, `.venv-inference` is the prepared environment. To expose it as a notebook kernel, install `ipykernel` in that environment and register/select that kernel. A fresh machine should use a new virtual environment; do not assume another machine has the local `.venv-inference` directory.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from collections import Counter

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "scripts/train_model.py").is_file(), "Set ROOT to your Vision-Drive repository folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)
print("Notebook interpreter:", sys.executable)


In [ ]:
# Run All defaults to inspecting existing artifacts, not downloading/retraining.
INSTALL_PACKAGES = False
SETUP_UPSTREAM = False
DOWNLOAD_DATA = False
TRAIN_MODEL = False
EVALUATE_MODEL = False

CLASSES = ["bottle", "cup", "box"]
DATASET = ROOT / "datasets/bottle-cup-box"
WEIGHTS = ROOT / "weights/obstacle/best.pt"
REPORT = ROOT / "docs/model-evaluation.json"
EPOCHS = 60
BATCH = 16
PER_CLASS_LIMIT = 250


## 2. Dependencies and upstream training initialization

Set `INSTALL_PACKAGES` to install the project's tested direct versions into **this kernel's interpreter**. Restart the kernel after changing PyTorch/NumPy packages.

`setup_model.py` retrieves the pinned Espressif source and initialization weights. Those upstream weights were originally trained for cats, but they are used **only to initialize transferable layers**. The training script rebuilds the head for the three target classes. The active application rejects the original cat checkpoint.

The initialization model and upstream source keep their original license/provenance; see `THIRD_PARTY_NOTICES.md`.


In [ ]:
if INSTALL_PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-model.txt")], check=True)
else:
    print("Package installation skipped.")
if SETUP_UPSTREAM:
    subprocess.run([sys.executable, "scripts/setup_model.py"], cwd=ROOT, check=True)


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
print("CPU-only training is supported. An NVIDIA GPU alone does not make a CPU wheel CUDA-enabled.")


## 3. Prepare labeled public images

The preparation script uses the official **Open Images validation source**, with Bottle, Coffee cup, and Box boxes mapped to the three project classes. It creates **new project splits**, deduplicates image content, and retains image IDs, authors, license/source URLs, and all selected target boxes.

The initial local run prepared 402 images: 280 training, 60 validation, and 62 test. Source availability limited boxes to 68 images. This is **not an official Open Images benchmark evaluation**. Downloading is opt-in here; data and intermediate files stay outside Git.


In [ ]:
if DOWNLOAD_DATA:
    if DATASET.exists():
        print("Dataset already exists; reusing it. Choose a new output directory to rebuild.")
    else:
        subprocess.run([sys.executable, "scripts/build_object_dataset.py", "--per-class", str(PER_CLASS_LIMIT),
                        "--output", str(DATASET)], cwd=ROOT, check=True)

manifest_path = DATASET / "manifest.jsonl"
if manifest_path.exists():
    records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines()]
    print("Image splits:", dict(Counter(row["split"] for row in records)))
    for split in ("train", "val", "test"):
        counts = Counter(obj["label"] for row in records if row["split"] == split for obj in row["objects"])
        print(split, "annotated instances:", dict(counts))
else:
    records = []
    print("No local dataset. Set DOWNLOAD_DATA=True to prepare it; stored model/report can still be inspected.")


## 4. Fine-tune ESPDet-Pico

Training uses 224 × 224 images, batch 16, seed 42, up to 60 epochs, and patience 15. The script constructs the upstream symbolic `nc` architecture and transfers compatible weights. It verifies that the head's class count matches the dataset.

The compatibility adapter handles the pinned ESP head's constructor with the project's Ultralytics version. Inference still uses the Espressif architecture; there is no generic-detector substitution. If memory is insufficient, lower `BATCH`. Validation selects the best checkpoint; test images are not used for that selection.


In [ ]:
if TRAIN_MODEL:
    assert (DATASET / "dataset.yaml").exists(), "Prepare the dataset first."
    subprocess.run([sys.executable, "scripts/train_model.py", "--data", str(DATASET / "dataset.yaml"),
                    "--epochs", str(EPOCHS), "--batch", str(BATCH), "--patience", "15",
                    "--device", DEVICE, "--output", str(ROOT / "runs/notebook-objects")], cwd=ROOT, check=True)
    candidates = list((ROOT / "runs").glob("notebook-objects*/weights/best.pt"))
    WEIGHTS = max(candidates, key=lambda path: path.stat().st_mtime)
    print("Trained checkpoint:", WEIGHTS)
else:
    print("Using existing published checkpoint:", WEIGHTS, "present:", WEIGHTS.is_file())


## 5. Inspect training curves

These curves show training/validation history, not test accuracy. A rising training score with a flat or falling validation score can indicate overfitting. The notebook reads either the fresh run or the repository's recorded training history.


In [ ]:
import matplotlib.pyplot as plt
history = WEIGHTS.parent.parent / "results.csv"
if not history.exists():
    history = ROOT / "docs/training-results.csv"
if history.exists():
    with history.open(newline="", encoding="utf-8-sig") as handle:
        rows = [{key.strip(): value for key, value in row.items()} for row in csv.DictReader(handle)]
    epochs = [float(row["epoch"]) for row in rows]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ("train/box_loss", "train/cls_loss"):
        axes[0].plot(epochs, [float(row[key]) for row in rows], label=key)
    for key in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        axes[1].plot(epochs, [float(row[key]) for row in rows], label=key)
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=.2)
    plt.tight_layout()
else:
    print("Training history is not available yet.")


## 6. Select confidence on validation, evaluate held-out test

The evaluation script selects a confidence threshold using **validation macro F1 only**, then reports class-wise precision/recall/F1 on untouched test images using IoU ≥ 0.5. This threshold-specific report is distinct from mAP. Missing labels in a public dataset and the small subset limit what can be concluded.

Set `EVALUATE_MODEL=True` to rerun this step with local images. Otherwise inspect the stored report. A real mini-car camera/course evaluation is still required.


In [ ]:
if EVALUATE_MODEL:
    REPORT = ROOT / "runs/notebook-evaluation.json"
    subprocess.run([sys.executable, "scripts/evaluate_objects.py", "--weights", str(WEIGHTS),
                    "--dataset", str(DATASET), "--output", str(REPORT)], cwd=ROOT, check=True)
if REPORT.exists():
    evaluation = json.loads(REPORT.read_text(encoding="utf-8"))
    print("Validation-selected confidence:", evaluation["confidence_threshold"])
    print("Test image count:", evaluation["image_counts"]["test"])
    for name, result in evaluation["test"]["per_class"].items():
        print(f"{name:8s} precision={result['precision']:.3f} recall={result['recall']:.3f} F1={result['f1']:.3f}")
else:
    evaluation = None
    print("Evaluation report is not available yet.")


## 7. Visualize a genuine model prediction and instruction

Set `EXAMPLE_IMAGE` to a local photo, or let the cell choose a held-out dataset image. It runs the real trained model and shows **detections → region overlap → rule candidate → instruction**. For a single still image, confirmation is reduced to one observation so the selected rule can be examined; the live app keeps its temporal confirmation and freshness checks.


In [ ]:
EXAMPLE_IMAGE = None  # e.g. ROOT / "my-bottle-photo.jpg"
if EXAMPLE_IMAGE is None and records:
    row = next(row for row in records if row["split"] == "test")
    EXAMPLE_IMAGE = DATASET / row["image"]
if EXAMPLE_IMAGE is not None and WEIGHTS.exists():
    import cv2
    from dataclasses import replace
    import time
    from visiondrive.config import Config
    from visiondrive.decision import Controller
    from visiondrive.detector import ESPDetector
    from visiondrive.explain import render_explanation
    config = replace(Config.load(ROOT / "config/default.json"), stable_frames=1, stale_after_ms=60000)
    detector = ESPDetector(WEIGHTS, ROOT / "external/esp-detection", config)
    detector.warmup()
    image = cv2.imread(str(EXAMPLE_IMAGE))
    assert image is not None, f"Cannot open {EXAMPLE_IMAGE}"
    start = time.monotonic()
    predictions = detector.predict(image)
    now = time.monotonic()
    decision = Controller(config).update(predictions, 0, start, now)
    canvas = render_explanation(image, decision, config, detector.name, {"status": "running"}, detector.label)
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
else:
    print("Set EXAMPLE_IMAGE and prepare the upstream source/checkpoint to run actual inference.")


## 8. Launch the live visualization and future hardware work

From a terminal in the repository, use `run_logic_demo.cmd` on Windows, or `python logic_demo.py` in a model environment. Use `--source path/to/video.mp4` for a recording. No camera is opened by this notebook automatically.

The detector recognizes only bottle/cup/box classes. An undetected object does not establish that the path is clear. The output is an instruction visualization, not physical-car control.

See [run instructions](../docs/laptop-esp32.md), [model report](../docs/model-report.md), and the [hardware implementation guide](../docs/hardware-implementation.md). Training on a laptop does not establish ESP32 memory use, speed, or quantized-model accuracy.
